<a href="https://colab.research.google.com/github/Aswanth0704/gpu-programming-cpp/blob/main/7_Asynchrony.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

What we have learned ealier:
- Execution spaces: where code runs (cuda: device or cpu: host)
- Memory spaces: Where data is stored (host vs device).
- Parallel algorithms (thrust::tabulate, thrust::transform, thrust::reduce, thrust::reduce_by_key)

This is how the improved simulator looks like now.

In [ ]:
%%writefile Sources/current_state.cpp
void simulate(int height, int width,
              thrust::device_vector<float> &in,
              thrust::device_vector<float> &out)
{
  // cuda mdspan to view it as a grid:
  cuda::std::mdspan temp_in(in.data(), height, width);
  thrust::tabulate(
    thrust::device, out.begin(), out.end(),
    [=] __host__ __device__ (int id)
    {
      /*..do something..*/
    }
  );
}

int main()
{
  // actually I need to initialize this, but let's skip that
  thrust::device_vector<float> d_prev(height * width);

  thrust::device_vector<float> d_next(height * width);
  thrust::host_vector<float> h_prev(height*width);

  for (int write_step = 0; write_step<3; write_step++)
  {
    // copy explicitly from GPU to Host:
    thrust::copy(d_prev.begin(), d_prev.end(), h_prev.begin());
    // store data to disk
    ach::store(write_step, height, width, h_prev);

    // compute next state on the GPU.
    for (int compute_step = 0; compute_step<3; compute_step++)
    {
      simulate(height, width, d_prev, d_next);
      d_prev.swap(d_next);
    }
  }
}

## Overlapping
Because thrust calls are synchronous, CPU has to wait for the GPU to complete the simulation before it can write the previous results to the disk. Writing efficient *heterogenous code* means, utilizing all the available resources, including the CPU. In many real world applications we can keep the CPU busy at the same time the GPU is computing. This is called *Overlapping*.

Unfortunately Thrust is synchronous and doesnot provide a means to separate launching GPU work from waiting for its completion. But, Thrust under the hood utilizes another library called CUB (CUDA unbound) to implement GPU algorithms,

## CUB
- gives finer control to utilize CPU while GPU kernels are still running. Its asynchronous.


In [1]:
import os

if os.getenv("COLAB_RELEASE_TAG"): # If running in Google Colab:
  !mkdir -p Sources
  !wget https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/02.02-Asynchrony/Sources/ach.h -nv -O Sources/ach.h
  !wget https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/02.02-Asynchrony/Sources/nvtx3.hpp -nv -O Sources/nvtx3.hpp

2026-09-27 17:08:27 URL:https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/02.02-Asynchrony/Sources/ach.h [5329/5329] -> "Sources/ach.h" [1]
2026-09-27 17:08:27 URL:https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/02.02-Asynchrony/Sources/nvtx3.hpp [105839/105839] -> "Sources/nvtx3.hpp" [1]


In [7]:
%%writefile Sources/cub-perf.cpp
#include "ach.h"

float simulate(int height, int width,
               const thrust::device_vector<float> &in,
               thrust::device_vector<float> &out,
               bool use_cub)
{
  // md span
  cuda::std::mdspan temp_in(thrust::raw_pointer_cast(in.data()), height, width);

  // compute
  auto compute = [=] __host__ __device__ (int id)
  {
    int row = id / width;
    int col = id % width;

    // loop over all the cells and update (except the boundary)
    if (row >0 && row < height - 1 && col>0 && col < width -1)
    {
      // evaluate derivatives:
      float d2tdx2 = temp_in(row, col-1) -2*temp_in(row, col) + temp_in(row, col + 1);
      float d2tdy2 = temp_in(row-1, col) -2*temp_in(row, col) + temp_in(row + 1, col);

      // update and return the temperatures:
      return temp_in(row, col) + 0.2f*(d2tdx2 + d2tdy2);
    }
    else
    {
      return temp_in(row, col);
    }

  };
  // time it:
  auto begin = std::chrono::high_resolution_clock::now();

  if (use_cub)
  {
    auto cell_ids = thrust::make_counting_iterator(0);
    cub::DeviceTransform::Transform(cell_ids, out.begin(), width*height, compute);
  }
  else
  {
    thrust::tabulate(thrust::device, out.begin(), out.end(), compute);
  }
  auto end = std::chrono::high_resolution_clock::now();
  return std::chrono::duration<float, std::milli>(end - begin).count();
}


int main()
{
  std::cout<<"size, thrust, cub\n";

  for(int size = 1024; size<= 16384; size*=2)
  {
    int width = size;
    int height = size;

    thrust::device_vector<float> current_temp(height*width, 15.0f);
    thrust::device_vector<float> next_temp(height*width);

    std::cout<<size<<", "
             <<simulate(height, width, current_temp, next_temp, false)
             <<", "
             <<simulate(height, width, current_temp, next_temp, true)
             <<std::endl;
  }
}

Overwriting Sources/cub-perf.cpp


In [8]:
!nvcc --extended-lambda -o /tmp/a.out Sources/cub-perf.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

size, thrust, cub
1024, 0.139176, 0.027546
2048, 0.179532, 0.005538
4096, 0.599305, 0.00553
8192, 2.38899, 0.024541
16384, 9.73663, 0.016151


From the above, I see that for thrust there is an increase in compute time as the size increases which makes sense.

But for cub, there is some irregularity. Since CUB is asynchronous, it launches the GPU kernels and then immediately returns control to the CPU. That means the time that you see are handover times not the actual times to for computation. We can correct this using cudaDeviceSynchronize(), to wait for the GPU to finish.

In [11]:
%%writefile Sources/cub-perf-sync.cpp
#include "ach.h"

float simulate(int height, int width,
               const thrust::device_vector<float> &in,
               thrust::device_vector<float> &out,
               bool use_cub)
{
  // md span
  cuda::std::mdspan temp_in(thrust::raw_pointer_cast(in.data()), height, width);

  // compute
  auto compute = [=] __host__ __device__ (int id)
  {
    int row = id / width;
    int col = id % width;

    // loop over all the cells and update (except the boundary)
    if (row >0 && row < height - 1 && col>0 && col < width -1)
    {
      // evaluate derivatives:
      float d2tdx2 = temp_in(row, col-1) -2*temp_in(row, col) + temp_in(row, col + 1);
      float d2tdy2 = temp_in(row-1, col) -2*temp_in(row, col) + temp_in(row + 1, col);

      // update and return the temperatures:
      return temp_in(row, col) + 0.2f*(d2tdx2 + d2tdy2);
    }
    else
    {
      return temp_in(row, col);
    }

  };
  // time it:
  auto begin = std::chrono::high_resolution_clock::now();

  if (use_cub)
  {
    auto cell_ids = thrust::make_counting_iterator(0);
    cub::DeviceTransform::Transform(cell_ids, out.begin(), width*height, compute);
    cudaDeviceSynchronize();
  }
  else
  {
    thrust::tabulate(thrust::device, out.begin(), out.end(), compute);
  }

  auto end = std::chrono::high_resolution_clock::now();
  return std::chrono::duration<float, std::milli>(end - begin).count();
}


int main()
{
  std::cout<<"size, thrust, cub\n";

  for(int size = 1024; size<= 16384; size*=2)
  {
    int width = size;
    int height = size;

    thrust::device_vector<float> current_temp(height*width, 15.0f);
    thrust::device_vector<float> next_temp(height*width);

    std::cout<<size<<", "
             <<simulate(height, width, current_temp, next_temp, false)
             <<", "
             <<simulate(height, width, current_temp, next_temp, true)
             <<std::endl;
  }
}

Overwriting Sources/cub-perf-sync.cpp


In [12]:
!nvcc --extended-lambda -o /tmp/a.out Sources/cub-perf-sync.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

size, thrust, cub
1024, 0.093394, 0.070546
2048, 0.15859, 0.158308
4096, 0.591078, 0.587385
8192, 2.2949, 2.29282
16384, 9.35612, 8.98168
